# 법제처 Open API로 AI 기본법 불러오기 테스트

- API 안내: https://open.law.go.kr/LSO/openApi/openApiManual.do
- 대상: 「인공지능 발전과 신뢰 기반 조성 등에 관한 기본법」 현행 (법률 제21311호, 2026.7.21 시행)
- 흐름: **목록조회**(`lawSearch.do`)로 법령일련번호(MST) 찾기 → **본문조회**(`lawService.do`)로 조문 가져오기
- 인증값 `OC`: `.env`의 `LAW_OC` (없으면 공용 샘플 `test`). 실사용은 open.law.go.kr에서 신청한 본인 OC 사용

# 1. 설정

In [1]:
import json
from pathlib import Path

import requests

from common.config import LAW_OC

BASE = "https://www.law.go.kr/DRF"
LAW_NAME = "인공지능 발전과 신뢰 기반 조성 등에 관한 기본법"
print("OC 설정됨:", bool(LAW_OC) and LAW_OC != "test")

OC 설정됨: True


# 2. 목록조회 → 현행 법률의 MST 찾기
같은 이름으로 검색하면 **시행령(대통령령)** 도 함께 나오므로 `법령구분명 == "법률"`로 거른다.

In [2]:
res = requests.get(f"{BASE}/lawSearch.do",
                   params={"OC": LAW_OC, "target": "law", "type": "JSON", "query": LAW_NAME},
                   timeout=10)
res.raise_for_status()
laws = res.json()["LawSearch"]["law"]
laws = laws if isinstance(laws, list) else [laws]  # 결과가 1건이면 dict로 옴

for l in laws:
    print(l["법령구분명"], l["법령명한글"], "| MST", l["법령일련번호"], "| 공포", l["공포번호"], "| 시행", l["시행일자"])

law = next(l for l in laws if l["법령구분명"] == "법률" and l["법령명한글"] == LAW_NAME)
MST = law["법령일련번호"]

법률 인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 | MST 282791 | 공포 21311 | 시행 20260721
대통령령 인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 시행령 | MST 288781 | 공포 36580 | 시행 20260820


# 3. 본문조회

In [3]:
res = requests.get(f"{BASE}/lawService.do",
                   params={"OC": LAW_OC, "target": "law", "type": "JSON", "MST": MST},
                   timeout=30)
res.raise_for_status()
data = res.json()["법령"]

info = data["기본정보"]
print(info["법령명_한글"], "| 법률 제", info["공포번호"], "호 | 공포", info["공포일자"])
assert info["공포번호"] == "21311", "현행 버전이 바뀌었는지 확인 필요"

인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 | 법률 제 21311 호 | 공포 20260120


# 4. 조문을 텍스트로 펼치기
API 응답 구조: `조문단위` → `항` → `호` → `목`
- `조문여부`가 `"전문"`이면 장 제목(예: 제1장 총칙), `"조문"`이면 실제 조문
- 항·호·목이 **1개면 dict, 여러 개면 list**로 오므로 항상 list로 맞춘다
- 조의2·의3은 `조문가지번호`로 구분 (예: 제17조의2)

In [4]:
def as_list(x):
    return x if isinstance(x, list) else ([x] if x else [])

def article_no(u):
    no = f"제{u['조문번호']}조"
    return no + f"의{u['조문가지번호']}" if u.get("조문가지번호") else no

def article_text(u):
    lines = [u["조문내용"].strip()]
    for hang in as_list(u.get("항")):
        if hang.get("항내용"):
            lines.append(hang["항내용"].strip())
        for ho in as_list(hang.get("호")):
            lines.append("  " + ho["호내용"].strip())
            for mok in as_list(ho.get("목")):
                lines.append("    " + mok["목내용"].strip())
    return "\n".join(lines)

units = data["조문"]["조문단위"]
articles, chapter = [], None
for u in units:
    if u["조문여부"] == "전문":
        chapter = u["조문내용"].strip()
        continue
    articles.append({
        "article": article_no(u),
        "title": u.get("조문제목", ""),
        "chapter": chapter,
        "content": article_text(u),
    })

print("조문 수:", len(articles))  # 현행 46개 = 제1~43조 + 제17조의2·제22조의2·제22조의3
assert len(articles) == 46

조문 수: 46


# 5. 일부 확인

In [5]:
by_no = {a["article"]: a for a in articles}

for no in ["제1조", "제31조", "제43조"]:
    a = by_no[no]
    print(f"[{a['chapter']}] {a['article']}({a['title']})")
    print(a["content"], end="\n\n")

[제1장 총칙] 제1조(목적)
제1조(목적) 이 법은 인공지능의 건전한 발전과 신뢰 기반 조성에 필요한 기본적인 사항을 규정함으로써 국민의 권익과 존엄성을 보호하고 국민의 삶의 질 향상과 국가경쟁력을 강화하는 데 이바지함을 목적으로 한다.

[제4장 인공지능윤리 및 신뢰성 확보] 제31조(인공지능 투명성 확보 의무)
제31조(인공지능 투명성 확보 의무)
① 인공지능사업자는 고영향 인공지능이나 생성형 인공지능을 이용한 제품 또는 서비스를 제공하려는 경우 제품 또는 서비스가 해당 인공지능에 기반하여 운용된다는 사실을 이용자에게 사전에 고지하여야 한다.
② 인공지능사업자는 생성형 인공지능 또는 이를 이용한 제품 또는 서비스를 제공하는 경우 그 결과물이 생성형 인공지능에 의하여 생성되었다는 사실을 표시하여야 한다.
③ 인공지능사업자는 인공지능시스템을 이용하여 실제와 구분하기 어려운 가상의 음향, 이미지 또는 영상 등의 결과물을 제공하는 경우 해당 결과물이 인공지능시스템에 의하여 생성되었다는 사실을 이용자가 명확하게 인식할 수 있는 방식으로 고지 또는 표시하여야 한다. 이 경우 해당 결과물이 예술적ㆍ창의적 표현물에 해당하거나 그 일부를 구성하는 경우에는 전시 또는 향유 등을 저해하지 아니하는 방식으로 고지 또는 표시할 수 있다.
④ 그 밖에 제1항에 따른 사전고지, 제2항에 따른 표시, 제3항에 따른 고지 또는 표시의 방법 및 그 예외 등에 관하여 필요한 사항은 대통령령으로 정한다.

[제6장 벌칙] 제43조(과태료)
제43조(과태료)
① 다음 각 호의 어느 하나에 해당하는 자에게는 3천만원 이하의 과태료를 부과한다.
  1. 제31조제1항을 위반하여 고지를 이행하지 아니한 자
  2. 제36조제1항을 위반하여 국내대리인을 지정하지 아니한 자
  3. 제40조제3항에 따른 중지명령이나 시정명령을 이행하지 아니한 자
② 제1항에 따른 과태료는 대통령령으로 정하는 바에 따라 과학기술정보통신부장관이 부과ㆍ징수한다.



In [6]:
# 2026.7.21 개정분이 들어왔는지: 신설 조문 + 제35조① 후단
print(by_no["제17조의2"]["content"], end="\n\n")
print(by_no["제35조"]["content"].splitlines()[1])
assert "인공지능취약계층" in by_no["제35조"]["content"]

제17조의2(인공지능제품 및 인공지능서비스 이용비용의 지원)
① 국가와 지방자치단체는 경제적 여건으로 인하여 인공지능제품 및 인공지능서비스를 이용하기 어려운 사람에 대하여 예산의 범위에서 그 비용의 전부 또는 일부를 지원할 수 있다.
② 제1항에 따른 비용 지원의 요건과 대상 등에 필요한 사항은 대통령령으로 정한다.

① 인공지능사업자가 고영향 인공지능을 이용한 제품 또는 서비스를 제공하는 경우 사전에 사람의 기본권에 미치는 영향을 평가(이하 "영향평가"라 한다)하기 위하여 노력하여야 한다. 이 경우 영향평가에는 고영향 인공지능을 이용한 제품 또는 서비스의 성격을 고려하여 인공지능취약계층의 특성이 반영될 수 있도록 하여야 한다. <개정 2026.1.20>


# 6. 원본 저장 (이후 청킹·임베딩에 사용)

In [7]:
out = Path("../data/ai_basic_law")
out.mkdir(parents=True, exist_ok=True)
(out / "law_raw.json").write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
(out / "articles.json").write_text(json.dumps(articles, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", sorted(p.name for p in out.iterdir()))

saved: ['articles.json', 'law_raw.json']
